In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'data' / '3_train.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'data' / '3_val.parquet')

# from sklearn import set_config
# set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = [
    col for col in (train.select_dtypes(include='number').columns)
    if col not in ['event_id', target_col]]  # 'station_id', 'timestamp'는 미포함
origin_cols = wl_cols + rain_cols + env_cols

assert len(num_cols) == 31, "'num_cols' 개수 이상"  # 종속변수 제외됨
assert (set(train.columns) - set(num_cols)) == {'station_id', 'timestamp', 'event_id', target_col}, "'num_cols' 확인필요"
assert set(origin_cols).issubset(num_cols), "'origin_cols' 확인필요"

In [3]:
# ============================================================
# 종속변수 분리
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [4]:
# ============================================================
# 모델1. 기본모델(주어진 수치형변수만으로, station_id 제외)
# ============================================================

# 독립변수 정의
X_train = train[origin_cols]
X_val = val[origin_cols]

# 조건별 rmse 저장용(누적)
results = []

# 기본모델 정의
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

base_model = LGBMRegressor(
    random_state=42
)

base_model.fit(X_train, y_train)

pred_base_val = base_model.predict(X_val)
pred_base_train = base_model.predict(X_train)

rmse_base_val = root_mean_squared_error(y_val, pred_base_val)
rmse_base_train = root_mean_squared_error(y_train, pred_base_train)

results.append({
    'Model': '1_base_model',
    'val_RMSE': rmse_base_val,
    'train_RMSE': rmse_base_train,
    'Gap': (rmse_base_val - rmse_base_train)
})

print(results[-1])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012018 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5525
[LightGBM] [Info] Number of data points in the train set: 616293, number of used features: 22
[LightGBM] [Info] Start training from score 0.111707
{'Model': '1_base_model', 'val_RMSE': 0.21528377032735027, 'train_RMSE': 0.23418222297586824, 'Gap': -0.01889845264851797}


In [5]:
# ============================================================
# 모델2. 모델1 + 수위차분변수
# ============================================================

# 독립변수
cols = origin_cols + wl_diff_cols
X_train_2 = train[cols]
X_val_2 = val[cols]

# 모델 학습
model2 = LGBMRegressor(random_state=42)
model2.fit(X_train_2, y_train)

# 예측
pred_train = model2.predict(X_train_2)
pred_val = model2.predict(X_val_2)

# 평가
rmse_train = root_mean_squared_error(y_train, pred_train)
rmse_val = root_mean_squared_error(y_val, pred_val)

# 결과 저장
results.append({
    'Model': '2_wl_diff',
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': rmse_val - rmse_train
})

print(results[-1])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008531 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6800
[LightGBM] [Info] Number of data points in the train set: 616293, number of used features: 27
[LightGBM] [Info] Start training from score 0.111707
{'Model': '2_wl_diff', 'val_RMSE': 0.20342969916046263, 'train_RMSE': 0.22019262128410255, 'Gap': -0.016762922123639917}


In [6]:
# ============================================================
# 모델3. 모델2 + 강우강도변수
# ============================================================
# val RMSE 기준 모델2 < 모델3: 모델2 선정

# 독립변수
cols = origin_cols + wl_diff_cols + rain_int_cols
X_train_3 = train[cols]
X_val_3 = val[cols]

# 모델 학습
model3 = LGBMRegressor(random_state=42)
model3.fit(X_train_3, y_train)

# 예측
pred_train = model3.predict(X_train_3)
pred_val = model3.predict(X_val_3)

# 평가
rmse_train = root_mean_squared_error(y_train, pred_train)
rmse_val = root_mean_squared_error(y_val, pred_val)

# 결과 저장
results.append({
    'Model': '3_rain_int',
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': rmse_val - rmse_train
})

print(results[-1])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016748 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7820
[LightGBM] [Info] Number of data points in the train set: 616293, number of used features: 31
[LightGBM] [Info] Start training from score 0.111707
{'Model': '3_rain_int', 'val_RMSE': 0.20376277743436555, 'train_RMSE': 0.2186143118497253, 'Gap': -0.014851534415359746}


In [7]:
# ============================================================
# 모델4s. 모델2(수위차분변수 추가) 변수군 ablation
# ============================================================

# 변수군 정의: for문 안에서 name, drop_cols 형태로 받을 수 있도록

feature_group = {
    'wl': wl_cols,
    'rain': rain_cols,
    'env': env_cols,
    'wl_diff': wl_diff_cols
}  # rain_int_cols는 기준모델에 없으므로 제외

num_cols2 = wl_cols + rain_cols + env_cols + wl_diff_cols  # rain_int_cols는 제외

# 모델 평가 함수 정의

def eval_model(cols):

    ablation_model = LGBMRegressor(random_state=42)

    ablation_model.fit(train[cols], y_train)

    pred_val = ablation_model.predict(val[cols])
    pred_train = ablation_model.predict(train[cols])

    rmse_val = root_mean_squared_error(y_val, pred_val)
    rmse_train = root_mean_squared_error(y_train, pred_train)

    return {
        'val_RMSE': rmse_val,
        'train_RMSE': rmse_train,
        'Gap': rmse_val - rmse_train
    }


# 모델링
for name, drop_cols in feature_group.items():  # (키, 값) 형태로 하나씩 반환

    cols = [col for col in num_cols2 if col not in drop_cols]

    score = eval_model(cols)

    results.append({
        'Model': f'no_{name}',
        **score  # eval_model(cols)의 return값 일체(딕셔너리 언패킹)
    })

    print(results[-1])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.051538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5270
[LightGBM] [Info] Number of data points in the train set: 616293, number of used features: 21
[LightGBM] [Info] Start training from score 0.111707
{'Model': 'no_wl', 'val_RMSE': 0.20652525026441396, 'train_RMSE': 0.22273479896874832, 'Gap': -0.01620954870433436}
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009534 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4544
[LightGBM] [Info] Number of data points in the train set: 616293, number of used features: 18
[LightGBM] [Info] Start training from score 0.111707
{'Model': 'no_rain', 'val_RMSE': 0.2127293070378692, 'train_RMSE': 0.24533872050522534, 'Gap': -0.03260941346735616}
[LightGBM] [Info]

In [8]:
# ============================================================
# 모델5. 모델2 + station_id OHE
# ============================================================

# 독립변수
cols = origin_cols + [c for c in wl_diff_cols if c != 'wl_t']
X_train_2 = train[cols]  # 모델2 그대로 가져온 것
X_val_2 = val[cols]

# 독립변수에 OHE 추가하기
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 안띄움
    sparse_output=True  # 희소행렬로 반환(원핫인코딩을 가볍게 만들어줌, 빅데이터에서 필수)
)

ohe.fit(train[['station_id']])  # skleran 인코더 2차원 행렬 필수

assert set(val['station_id']).issubset(set(train['station_id'])), "val에 변환되지 않은 'station_id' 발생 가능"

train_ohe = ohe.transform(train[['station_id']])
val_ohe = ohe.transform(val[['station_id']])
# 여기까지는 변환, 실제 데이터에 합친건 아님

# 합치기
from scipy.sparse import hstack

X_train_4 = hstack([
    X_train_2,
    train_ohe
]).tocsr()

X_val_4 = hstack([
    X_val_2,
    val_ohe
]).tocsr()

# 행 개수 검증
assert X_train_4.shape[0] == len(train), "Train 행 개수 불일치"
assert X_val_4.shape[0] == len(val), "Val 행 개수 불일치"
# 열 개수 검증
assert X_train_4.shape[1] == len(cols) + train_ohe.shape[1], "Train 열 개수 불일치"
assert X_val_4.shape[1] == X_train_4.shape[1], "Train과 Val 열 개수 불일치"

In [12]:
print(X_train_4.shape, X_val_4.shape)

(616293, 492) (133316, 492)


In [9]:
# ============================================================
# 모델5. 모델2 + station_id OHE
# ============================================================


# 모델 학습
model4 = LGBMRegressor(random_state=42)
model4.fit(X_train_4, y_train)

# 예측
pred_train = model4.predict(X_train_4)
pred_val = model4.predict(X_val_4)

# 평가
rmse_train = root_mean_squared_error(y_train, pred_train)
rmse_val = root_mean_squared_error(y_val, pred_val)

# 결과 저장
results.append({
    'Model': '4_station_ohe',
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': rmse_val - rmse_train
})

print(results[-1])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011674 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7730
[LightGBM] [Info] Number of data points in the train set: 616293, number of used features: 492
[LightGBM] [Info] Start training from score 0.111707
{'Model': '4_station_ohe', 'val_RMSE': 0.20244167692861822, 'train_RMSE': 0.2194754923477144, 'Gap': -0.017033815419096182}


In [11]:
print(results)

df_results = pd.DataFrame(results)
print(df_results)

[{'Model': '1_base_model', 'val_RMSE': 0.21528377032735027, 'train_RMSE': 0.23418222297586824, 'Gap': -0.01889845264851797}, {'Model': '2_wl_diff', 'val_RMSE': 0.20342969916046263, 'train_RMSE': 0.22019262128410255, 'Gap': -0.016762922123639917}, {'Model': '3_rain_int', 'val_RMSE': 0.20376277743436555, 'train_RMSE': 0.2186143118497253, 'Gap': -0.014851534415359746}, {'Model': 'no_wl', 'val_RMSE': 0.20652525026441396, 'train_RMSE': 0.22273479896874832, 'Gap': -0.01620954870433436}, {'Model': 'no_rain', 'val_RMSE': 0.2127293070378692, 'train_RMSE': 0.24533872050522534, 'Gap': -0.03260941346735616}, {'Model': 'no_env', 'val_RMSE': 0.21689196678148784, 'train_RMSE': 0.2344002814264954, 'Gap': -0.017508314645007572}, {'Model': 'no_wl_diff', 'val_RMSE': 0.21528377032735027, 'train_RMSE': 0.23418222297586824, 'Gap': -0.01889845264851797}, {'Model': '4_station_ohe', 'val_RMSE': 0.20244167692861822, 'train_RMSE': 0.2194754923477144, 'Gap': -0.017033815419096182}]
           Model  val_RMSE  tra